In [1]:
!pip install -q evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
import evaluate

In [3]:
# 1. Load and preprocess dataset
df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")

print(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


                                               image              caption  \
0  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...       Tomato healthy   
1  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   Tomato Late blight   
2  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...       Tomato healthy   
3  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  Tomato mosaic virus   
4  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  Pepper bell healthy   

                                            captions  
0  [A vibrant green and healthy tomato leaf with ...  
1  [A tomato leaf showing dark brown lesions and ...  
2  [A vibrant green and healthy tomato leaf with ...  
3  [A tomato leaf with mosaic-like patterns of li...  
4  [A fresh green bell pepper leaf with a smooth,...  


In [4]:
df = df.drop(columns=["image"])
print(df.head())
print(df.shape)


               caption                                           captions
0       Tomato healthy  [A vibrant green and healthy tomato leaf with ...
1   Tomato Late blight  [A tomato leaf showing dark brown lesions and ...
2       Tomato healthy  [A vibrant green and healthy tomato leaf with ...
3  Tomato mosaic virus  [A tomato leaf with mosaic-like patterns of li...
4  Pepper bell healthy  [A fresh green bell pepper leaf with a smooth,...
(20638, 2)


In [5]:
df = df.explode("captions")
print(df.head())
print(df.shape)

              caption                                           captions
0      Tomato healthy  A vibrant green and healthy tomato leaf with s...
0      Tomato healthy  A healthy Solanum lycopersicum leaf, free of d...
0      Tomato healthy  A fresh tomato leaf outdoors, glowing in sunli...
0      Tomato healthy  A clean and healthy tomato leaf image, perfect...
1  Tomato Late blight  A tomato leaf showing dark brown lesions and w...
(82552, 2)


In [6]:

df.to_csv("dataset.csv", index=False)
df = pd.read_csv("dataset.csv")
print(df.head())

              caption                                           captions
0      Tomato healthy  A vibrant green and healthy tomato leaf with s...
1      Tomato healthy  A healthy Solanum lycopersicum leaf, free of d...
2      Tomato healthy  A fresh tomato leaf outdoors, glowing in sunli...
3      Tomato healthy  A clean and healthy tomato leaf image, perfect...
4  Tomato Late blight  A tomato leaf showing dark brown lesions and w...


In [7]:

# 2. Label encoding
num_labels = 15
data_col_name = "captions"
label_col_name = "caption"

encoder = LabelEncoder()
encoder.fit(df[label_col_name].tolist())
df["label"] = encoder.transform(df[label_col_name].tolist())

In [8]:

# 3. Train-test split
df_train, df_test = train_test_split(df, train_size=0.8, random_state=42)

train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

In [9]:
# 4. Tokenization
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(data):
    return tokenizer(data["captions"], truncation=True)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

In [10]:

# 5. Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [11]:
# 6. Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# 7. Evaluation metrics
eval_metrics = evaluate.load("accuracy")


In [13]:
def metrics(eval_pred):
    logits, labels = eval_pred
    prediction = np.argmax(logits, axis=1)
    return eval_metrics.compute(predictions=prediction, references=labels)


In [14]:
# 8. Training arguments
training_arguments = TrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=5e-5,
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=2,
    weight_decay=0.01,
    report_to="none"
)

In [15]:
# 9. Trainer setup
trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=metrics
)

/tmp/ipython-input-4227528030.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
# 10. Train the model
trainer.train()


Step,Training Loss
8256,0.022400
16512,0.000000


TrainOutput(global_step=16512, training_loss=0.011194817281281666, metrics={'train_runtime': 673.6905, 'train_samples_per_second': 196.057, 'train_steps_per_second': 24.51, 'total_flos': 922152682033560.0, 'train_loss': 0.011194817281281666, 'epoch': 2.0})

In [17]:
# 11. Save the model
model.save_pretrained("disease_detection_model")
tokenizer.save_pretrained("disease_detection_model")
print("Model and tokenizer saved successfully")


Model and tokenizer saved successfully


In [18]:
# 12. Apply model on disease detection
from transformers import pipeline

disease_detector = pipeline("text-classification", model="disease_detection_model", tokenizer="disease_detection_model")

sample_text = "This leaf shows yellow spots with brown edges and curling"
result = disease_detector(sample_text)

print("Prediction result:", result)

predicted_label = encoder.inverse_transform([int(result[0]['label'].split('_')[-1])])[0]
print("Predicted disease name:", predicted_label)

Device set to use cuda:0


Prediction result: [{'label': 'LABEL_8', 'score': 0.6101467609405518}]
Predicted disease name: Tomato Leaf Mold


In [19]:
import joblib

# 'encoder' is the LabelEncoder you fit in cell #2
joblib.dump(encoder, 'text_label_encoder.joblib')

print("Text label encoder saved to text_label_encoder.joblib")

Text label encoder saved to text_label_encoder.joblib


In [20]:
!zip -r disease_detection_model.zip disease_detection_model

  adding: disease_detection_model/ (stored 0%)
  adding: disease_detection_model/tokenizer.json (deflated 71%)
  adding: disease_detection_model/model.safetensors (deflated 8%)
  adding: disease_detection_model/tokenizer_config.json (deflated 75%)
  adding: disease_detection_model/vocab.txt (deflated 53%)
  adding: disease_detection_model/special_tokens_map.json (deflated 42%)
  adding: disease_detection_model/config.json (deflated 60%)
